# CALLIC speedup benchmark — test every suggestion, keep what wins
Each config runs the same timed protocol (`tools/bench.py::run_cfg`): identical seed/data, steady-state s/step + loss start→end. Winner = most patches/s at equal-or-better loss drop.
Needs GPU for compile/AMP rows (CPU runs only the portable subset).
> *Unofficial reproduction — code largely AI-generated; verify before trusting.*


In [ ]:
import os, sys
REPO_URL = 'https://github.com/hassenhamdi/callic.git'
def _find_repo():
    for c in [os.getcwd(), '/content']:
        root = c if os.path.basename(c) != 'callic' else os.path.dirname(c)
        if os.path.isdir(os.path.join(root, 'callic')):
            return root
    return None
ROOT = _find_repo()
if ROOT is None:
    import subprocess
    subprocess.run(['git', 'clone', REPO_URL, '/content/callic'], check=True)
    ROOT = '/content/callic'
    assert os.path.isdir(os.path.join(ROOT, 'callic')), f'clone lacks callic/ (got {os.listdir(ROOT)[:10]})'
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('repo root:', ROOT)
import os, sys
import torch
from tools.bench import BenchData, run_cfg, SYSTEM_PRESETS
try:
    import torch_xla.core.xla_model as xm
    DEV = 'xla'
except ImportError:
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV)
data = BenchData('data/DIV2K_valid_HR', n_patches=2048).pin()
print('patches:', data.data.shape, '| src:', data.source)


In [ ]:
# 0b. Official optimizer installs (session-scoped; re-run per fresh runtime)
# Muon (KellerJordan) + NorMuon pip-git; Aurora has no PyPI -> git clone, auto-pathed.
import os, subprocess, sys
os.makedirs('thirdparty', exist_ok=True)
if not os.path.isdir('thirdparty/aurora-release'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/tilde-research/aurora-release.git',
                    'thirdparty/aurora-release'], check=True)
    print('aurora-release cloned')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'git+https://github.com/KellerJordan/Muon',
                'git+https://github.com/zichongli5/NorMuon.git'])
sys.path.insert(0, 'thirdparty/aurora-release/src')  # aurora has no PyPI package
from aurora import aurora as _a
import inspect
print('aurora OK:', inspect.signature(_a))
from muon import SingleDeviceMuonWithAuxAdam as _M
from normuon import SingleDeviceNorMuonWithAuxAdam as _N
print('muon + normuon OK')


## A. System toggles (per-step wall-clock)
Baseline vs cudnn.benchmark, matmul-high, compile modes, fused model+loss NLL graph, all-combined.

In [ ]:
def show(rows):
    hdr = ['cfg', 's/step', 'patches/s', 'loss0->lossN']
    print(f"{hdr[0]:<18}{hdr[1]:<10}{hdr[2]:<12}{hdr[3]}")
    for name, r in rows:
        print(f"{name:<18}{r['s_per_step']:<10}{r['patches_per_s']:<12}{r['loss0']}->{r['lossN']}")
rows = []
for name, kw in SYSTEM_PRESETS:
    use = dict(kw)
    if DEV != 'cuda' and (use.get('compile_mode') or use.get('cudnn_bench')):
        print(f'{name}: skipped (CUDA-only)'); continue
    r = run_cfg(data, steps=30, bs=(128 if DEV == 'xla' else 32), device=DEV, **{k: v for k, v in use.items()})
    rows.append((name, r))
    print(name, '->', r['s_per_step'], 's/step,', r['patches_per_s'], 'patches/s', flush=True)
show(rows)


## B. Batch-size sweep (patches/s decides — earlier T4 data says bs32 wins, re-check on your GPU)

In [ ]:
rows = []
for bs in [32, 64, 128]:
    try:
        r = run_cfg(data, steps=20, bs=bs, device=DEV)
        rows.append((f'bs{bs}', r))
    except RuntimeError as e:
        print(f'bs{bs}: OOM ({str(e)[:80]})')
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
show(rows)


## C. Optimizer shootout (sample efficiency — fewer steps to same loss)
Muon-lite recipe: Muon on ≥2D (conv4D reshaped), AdamW on biases/norms/head; normuon adds row-RMS norm; turbo uses AOL init + 4 NS steps. 100 steps each; compare lossN.

In [ ]:
rows = []
cfgs = [('adamw', {}), ('adamw+warmup', {'warmup': 10}),
        ('muon', {'opt': 'muon'}), ('normuon', {'opt': 'normuon'}), ('aurora', {'opt': 'aurora'})]
for name, kw in cfgs:
    r = run_cfg(data, steps=100, bs=32, device=DEV, **kw)
    rows.append((name, r))
    print(name, '-> loss', r['loss0'], '->', r['lossN'], '@', r['s_per_step'], 's/step', flush=True)
show(rows)


## Decide
- System winner → bake into `tools/train.py` flags.
- Optimizer winner → full 2M run with it (same model, fidelity intact).
- Full Aurora (damped alternating iteration) only if Muon-family wins and tall-`W_up` death is suspected.

## D. TPU notes (v5e/v5p expectations)
- Device support is built in (`device='xla'`, bf16 autocast, XLA sync per step); CUDA-only rows auto-skip.
- Peak flops flatter small convs: v5e ≈3× T4, v5p ≈7× T4 per chip — but our 128-ch convs + depthwise + transcendental NLL underuse the MXUs. Realistic single-chip estimate: **1–2× a T4**, only with big batches (bs128–512 + scaled LR) and static shapes (no recompiles).
- XLA compile cost amortizes over 2M steps; per-step `loss.item()` syncs would kill it — logging stays every 500 steps.
- No TPU access in Lightning/free-Colab tiers (Colab TPUv2 is older than T4-class for this); needs GCP/TRC. GPU path remains the recommendation on a fixed budget.